In [ ]:
import os
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import matplotlib.pyplot as plt

def load_env_file(env_path=".env"):
    env_file = Path(env_path)
    if not env_file.exists():
        return
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value

load_env_file()
API_KEY = os.getenv("SCRAPERAPI_KEY")

if not API_KEY:
    raise ValueError("SCRAPERAPI_KEY not found. Add it in .env file.")

def get_html(url):
    api_url = "http://api.scraperapi.com"
    params = {
        "api_key": API_KEY,
        "url": url,
        "render": "false",
        "country_code": "in"
    }
    res = requests.get(api_url, params=params, timeout=30)
    return res.text

In [6]:
def scrape_99acres(city_code, city_name, pages=3):
    all_rows = []

    for page in range(1, pages+1):
        url = f"https://www.99acres.com/property-in-{city_code}-ffid-page-{page}"
        html = get_html(url)
        soup = BeautifulSoup(html, "html.parser")

        cards = soup.select("div[data-label='SEARCH']")
        if not cards:
            cards = soup.select("div.srpTuple__tuple")

        for c in cards:
            try:
                t = c.select_one("a.srpTuple__propertyName") or c.select_one("a")
                title = t.get_text(strip=True) if t else "NA"

                p = c.select_one("div.srpTuple__price") or c.select_one("span")
                price = p.get_text(strip=True) if p else ""

                l = c.select_one("div.srpTuple__location") or c.select_one("span")
                location = l.get_text(strip=True) if l else city_name

                link = ""
                if t and t.has_attr("href"):
                    link = "https://www.99acres.com" + t["href"]

                all_rows.append({
                    "City": city_name,
                    "Title": title,
                    "Price": price,
                    "Location": location,
                    "Link": link
                })
            except:
                continue

        time.sleep(1.5)

    return all_rows

In [7]:
delhi = scrape_99acres("delhi-ncr", "Delhi", pages=3)
mumbai = scrape_99acres("mumbai", "Mumbai", pages=3)

df = pd.DataFrame(delhi + mumbai)

print("Total rows:", len(df))
df.head()

Total rows: 3


,City,Title,Price,Location,Link
0,Delhi,Kohli East Delhi Luxury Builder Floors,"Ready To Move ·Since Jan, 2026","Ready To Move ·Since Jan, 2026",https://www.99acres.comhttps://www.99acres.com...
1,Mumbai,"2 BHK FlatinPokhran 2, Thane",2 people shortlisted this property,2 people shortlisted this property,https://www.99acres.comhttps://www.99acres.com...
2,Mumbai,Swastik Iris,"New Launch ·Completion in Jun, 2029","New Launch ·Completion in Jun, 2029",https://www.99acres.comhttps://www.99acres.com...


In [8]:
import re
import pandas as pd

def parse_price(x):
    try:
        if pd.isna(x):
            return (None, None, None)

        x = str(x).lower().replace(",", "").strip()

        # ❌ Skip invalid
        if "request" in x or "call" in x:
            return (None, None, None)

        # 🔥 Extract number (first number)
        match = re.search(r"\d+(\.\d+)?", x)
        if not match:
            return (None, None, None)

        value = float(match.group())

        # 🔥 Detect unit
        if "cr" in x or "crore" in x:
            return (value, "Cr", value * 1e7)

        elif "lac" in x or "lakh" in x:
            return (value, "Lakh", value * 1e5)

        else:
            return (value, "Raw", value)

    except:
        return (None, None, None)

In [9]:
parsed = df["Price"].apply(parse_price)

df_parsed = pd.DataFrame(parsed.tolist(), columns=["Price_Value", "Price_Unit", "Price_INR"])

df = pd.concat([df, df_parsed], axis=1)

df = df.dropna(subset=["Price_INR"])

In [10]:
df["Price"].head(20)

0         Ready To Move  ·Since Jan, 2026
1      2 people shortlisted this property
2    New Launch  ·Completion in Jun, 2029
Name: Price, dtype: object

In [11]:
for p in df["Price"].head(10):
    print(p, "->", parse_price(p))

Ready To Move  ·Since Jan, 2026 -> (2026.0, 'Raw', 2026.0)
2 people shortlisted this property -> (2.0, 'Raw', 2.0)
New Launch  ·Completion in Jun, 2029 -> (2029.0, 'Raw', 2029.0)


In [12]:
df.to_csv("99acres_clean_data.csv", index=False)
print("CSV saved successfully!")

CSV saved successfully!
